
RAG Pipelines- Data Ingestion to Vector DB Pipeline


In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\girin\AppData\Local\Temp\ipykernel_22656\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\girin\OneDrive\Desktop\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: rag_data.pdf
  ✓ Loaded 51 pages

Total documents loaded: 51


In [8]:
all_pdf_documents


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-06-02T17:21:52+05:30', 'author': '', 'moddate': '2026-06-02T17:21:52+05:30', 'title': 'Microsoft Word - 2110 E & IT Deptt.docx', 'source': '..\\data\\pdf\\rag_data.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1', 'source_file': 'rag_data.pdf', 'file_type': 'pdf'}, page_content='EXTRAORDINARY \n                                                PUBLISHED BY AUTHORITY \n  \n            No.2110,  CUTTACK,    TUESDAY,   MAY    26,   2026 /  JAISTHA   5,    1948   \n \n       [No.2729─PT1-EIT-SCH-I-EGCONF-0003/2019/E&IT.]   \n      ELECTRONICS & INFORMATION TECHNOLOGY DEPARTMENT \nRESOLUTION \n     The 25th May, 2026 \nOdisha State Data Policy-2.0 \nCONTENTS \nGlossary of Terms .................................................................................................. 5  \n1    Preamble ................................................................................................

In [9]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [10]:
chunks=split_documents(all_pdf_documents)
chunks

Split 51 documents into 149 chunks

Example chunk:
Content: EXTRAORDINARY 
                                                PUBLISHED BY AUTHORITY 
  
            No.2110,  CUTTACK,    TUESDAY,   MAY    26,   2026 /  JAISTHA   5,    1948   
 
       [No.2729─PT...
Metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-06-02T17:21:52+05:30', 'author': '', 'moddate': '2026-06-02T17:21:52+05:30', 'title': 'Microsoft Word - 2110 E & IT Deptt.docx', 'source': '..\\data\\pdf\\rag_data.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1', 'source_file': 'rag_data.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-06-02T17:21:52+05:30', 'author': '', 'moddate': '2026-06-02T17:21:52+05:30', 'title': 'Microsoft Word - 2110 E & IT Deptt.docx', 'source': '..\\data\\pdf\\rag_data.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1', 'source_file': 'rag_data.pdf', 'file_type': 'pdf'}, page_content='EXTRAORDINARY \n                                                PUBLISHED BY AUTHORITY \n  \n            No.2110,  CUTTACK,    TUESDAY,   MAY    26,   2026 /  JAISTHA   5,    1948   \n \n       [No.2729─PT1-EIT-SCH-I-EGCONF-0003/2019/E&IT.]   \n      ELECTRONICS & INFORMATION TECHNOLOGY DEPARTMENT \nRESOLUTION \n     The 25th May, 2026 \nOdisha State Data Policy-2.0 \nCONTENTS \nGlossary of Terms .................................................................................................. 5  \n1    Preamble ................................................................................................

Embedding And vectorStoreDB

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1253.74it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\girin\AppData\Local\Temp\ipykernel_22656\1472052616.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [13]:
class VectorStore:
    """Manages document embeddings in a DB Vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the chromaDB collection.
            persist_directory: Directory to persist the vector store data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            #create persist chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            #Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF documents embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in the collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of Document objects.
            embeddings: Numpy array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embeddings.")
        
        print(f"Adding {len(documents)} documents to the vector store...")
        
        ##Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generate unique id 
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            #Document content
            documents_texts.append(doc.page_content)
            
            #Embedding
            embeddings_list.append(embedding.tolist())
            
        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection after addition: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vectorstore = VectorStore()
vectorstore

Vector store initialized with collection: pdf_documents
Existing documents in the collection: 149


In [14]:
chunks


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-06-02T17:21:52+05:30', 'author': '', 'moddate': '2026-06-02T17:21:52+05:30', 'title': 'Microsoft Word - 2110 E & IT Deptt.docx', 'source': '..\\data\\pdf\\rag_data.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1', 'source_file': 'rag_data.pdf', 'file_type': 'pdf'}, page_content='EXTRAORDINARY \n                                                PUBLISHED BY AUTHORITY \n  \n            No.2110,  CUTTACK,    TUESDAY,   MAY    26,   2026 /  JAISTHA   5,    1948   \n \n       [No.2729─PT1-EIT-SCH-I-EGCONF-0003/2019/E&IT.]   \n      ELECTRONICS & INFORMATION TECHNOLOGY DEPARTMENT \nRESOLUTION \n     The 25th May, 2026 \nOdisha State Data Policy-2.0 \nCONTENTS \nGlossary of Terms .................................................................................................. 5  \n1    Preamble ................................................................................................

In [15]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 149 texts...


Batches: 100%|██████████| 5/5 [00:07<00:00,  1.51s/it]


Generated embeddings with shape: (149, 384)
Adding 149 documents to the vector store...
Successfully added 149 documents to the vector store.
Total documents in the collection after addition: 298


Retriever Pipeline From VectorStore

In [16]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [17]:
rag_retriever

In [18]:
rag_retriever.retrieve("What are the guiding principles of Odisha State Data Policy 2.0?")

Retrieving documents for query: 'What are the guiding principles of Odisha State Data Policy 2.0?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_7fccbf1b_53',
  'content': 'best practices for addressing the unique data driven challenges and opportunities in the state \nof Odisha. The policy also aims to implement capacity-building initiatives to equip government \nofficials with the necessary skills and knowledge to effectively manage and utilize data \nresources. \nVision: \nOSDP 2.0 aims to fundamentally change how Government of Odisha uses publicly \ncollected data from various sectors to drive change across society. This will be achieved by \nenhancing digital skills and data management capacities at all levels of administration to \nenable policymaking, improve implementation and monitoring of welfare programs and ensure \nefficient delivery of services for citizens. \nThe policy envisions a comprehensive framework for data governance that establishes \nclear standards and responsibilities for data management, ensures data security and privacy \nBeing a spearhead in leveraging technology for effective governan

In [19]:
rag_retriever.retrieve("What is the mission of Odisha State Data Policy 2.0?")

Retrieving documents for query: 'What is the mission of Odisha State Data Policy 2.0?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.80it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_7fccbf1b_53',
  'content': 'best practices for addressing the unique data driven challenges and opportunities in the state \nof Odisha. The policy also aims to implement capacity-building initiatives to equip government \nofficials with the necessary skills and knowledge to effectively manage and utilize data \nresources. \nVision: \nOSDP 2.0 aims to fundamentally change how Government of Odisha uses publicly \ncollected data from various sectors to drive change across society. This will be achieved by \nenhancing digital skills and data management capacities at all levels of administration to \nenable policymaking, improve implementation and monitoring of welfare programs and ensure \nefficient delivery of services for citizens. \nThe policy envisions a comprehensive framework for data governance that establishes \nclear standards and responsibilities for data management, ensures data security and privacy \nBeing a spearhead in leveraging technology for effective governan